# Antarctic lateral drag — manuscript analysis

This is the main publication notebook on `CICE_free-slip/develop`. It contains the experiment inventory, processing recipes, FIA comparisons, uncertainty/coverage checks and manuscript export record. `LD-performance.ipynb` contains the complementary spatial, mechanics, thickness and pack checks. Both use `ld_pub_config.json` and the tested reductions in `ld_pub_workflow.py`.

The full model diagnostic period remains **2000–2005**, after 1995–1999 spin-up. The common observational comparison is **1 April 2000–30 April 2005** (the attached April-to-April period interpreted as inclusive calendar months). Older panels extending to December 2005 must be regenerated. This is a period choice, not a claim of verified observation coverage.

The three additional controls are submitted, with results pending. The default run inventories inputs and writes recipes only; it does not submit jobs or invent results when a store is absent. Enable each analysis stage after checking its inputs, and run expensive reductions in a PBS allocation. Cleared outputs are intentional.

## 0. Shared configuration and loaders

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shlex
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = next((p for p in [Path.cwd(), Path.cwd()/'notebooks',
    Path(os.environ.get('CICE_FREE_SLIP_REPO', '.'))/'notebooks']
    if (p/'ld_pub_workflow.py').is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Start in the repository/notebooks directory, or set CICE_FREE_SLIP_REPO')
sys.path.insert(0, str(NOTEBOOK_DIR))
import ld_pub_workflow as wf
SHUGA_REPO = os.environ.get('SHUGA_REPO')
if SHUGA_REPO:
    shuga_repo = Path(SHUGA_REPO).expanduser()
    sys.path.insert(0, str(shuga_repo.parent if (shuga_repo/'__init__.py').exists() else shuga_repo))
import shuga
from shuga.core.types import RunSpec, ClassificationSpec, MetricsSpec
from shuga.core.paths import ShugaPaths
from shuga.core.regions import ANTARCTIC_8_REGIONS

CONFIG_PATH = Path(os.environ.get('LD_PUB_CONFIG', NOTEBOOK_DIR/'ld_pub_config.json'))
CFG = json.loads(CONFIG_PATH.read_text())
START, END = CFG['model_start'], CFG['model_end']
OBS_START, OBS_END = CFG['evaluation_start'], CFG['evaluation_end']
GROUP, METHOD = CFG['experiment_group'], CFG['method']
BANK = Path(CFG['afim_output_root'])
ARCHIVE = BANK
EXPORT = Path(os.environ.get('LD_PUB_EXPORT', str(Path(CFG['graphics_root'])/GROUP/'publication')))
EXPORT.mkdir(parents=True, exist_ok=True)
CACHE = EXPORT/'cache'; CACHE.mkdir(exist_ok=True)
manifest = wf.expected_experiments()
EXPERIMENTS = manifest.experiment.tolist()
FAMILIES = {family: part.experiment.tolist() for family, part in manifest.groupby('family', sort=False)}
REFERENCE = 'no-slip-LFI-match'
PRIMARY = ['no-slip-LFI-match','free-slip-no-drag','Cs-high','Cq-high']
TENSILE_CASES = ['Cs-high-ktens-zero','Cs-high-ktens-low','Cs-high-ktens-mid','Cs-high']
DIAGNOSTIC_CASES = list(dict.fromkeys(PRIMARY+TENSILE_CASES))
PAIRS = {'boundary_only':('free-slip-no-drag','no-slip-LFI-match'),
         'static_drag':('Cs-high','free-slip-no-drag'),
         'quadratic_drag':('Cq-high','free-slip-no-drag'),
         'tensile_term':('Cs-high','Cs-high-ktens-zero'),
         'coast_increment_over_gib':('Cs-high','GIB-drag'),
         'gib_increment_over_coast':('Cs-high','cst-drag')}
REGIONS = list(ANTARCTIC_8_REGIONS)
OBS_FIA_PATH = Path(CFG['obs_fia_path'])
UNIT_OVERRIDES = {}  # Only documented metadata corrections; never inferred from values.
OPEN, COVERAGE, ARTIFACTS = {}, [], []

def contexts(name, method=METHOD, ice_type='FI'):
    run=RunSpec(name,START,END,project=CFG['project'],user=CFG['user'],experiment_group=GROUP)
    cls=ClassificationSpec(ice_type=ice_type,methods=method,grid_type=CFG['grid_type'],
        ispd_thresh=CFG['ispd_thresh'],aice_thresh=CFG['aice_thresh'],
        bin_window=CFG['bin_window'],bin_min_days=CFG['bin_min_days'],roll_window=CFG['roll_window'])
    met=MetricsSpec(methods=method)
    paths=ShugaPaths(run_cfg=run,cls_cfg=cls,met_cfg=met,afim_output_root=BANK,
                     archive_root=ARCHIVE,graphics_root=CFG['graphics_root'])
    return dict(run_cfg=run,cls_cfg=cls,met_cfg=met,pth_cfg=paths)

def load(name,kind,variables,method=METHOD,ice_type='FI'):
    key=(name,kind,tuple(variables),method,ice_type)
    if key not in OPEN:
        kwargs=contexts(name,method,ice_type)
        if kind!='history': kwargs['classification']=method
        loader={'metrics':shuga.load_metrics,'mask':shuga.load_classified,'history':shuga.load_cice}[kind]
        ds=loader(**kwargs,variables=variables,dt0_str=START,dtN_str=END)
        missing=set(variables)-set(ds.variables)
        if missing: ds.close(); raise KeyError(f'{name}: missing {missing}')
        OPEN[key]=ds
    return OPEN[key]

def open_product(path):
    path=Path(path)
    return xr.open_zarr(path,consolidated=False) if path.suffix=='.zarr' else xr.open_dataset(path)

def audited(da,name):
    da=da.copy(deep=False)
    if (name,da.name) in UNIT_OVERRIDES:
        da.attrs=dict(da.attrs,units=UNIT_OVERRIDES[(name,da.name)])
    return da

def git_commit(path):
    try:return subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):return 'UNVERIFIED'

provenance=dict(config=CFG,helper_sha256=wf.file_sha256(NOTEBOOK_DIR/'ld_pub_workflow.py'),
    model_repository_commit=git_commit(NOTEBOOK_DIR.parent),shuga_module=shuga.__file__,
    shuga_commit=git_commit(Path(shuga.__file__).parent),unit_overrides={str(k):v for k,v in UNIT_OVERRIDES.items()})
SIGNATURE=wf.signature(provenance)

def register(path,diagnostic,cases,period=None,notes=''):
    period=period or (OBS_START,OBS_END)
    ARTIFACTS.append(dict(path=str(path),diagnostic=diagnostic,experiments=list(cases),
        start=period[0],end=period[1],method=METHOD,signature=SIGNATURE,notes=notes))

def export_table(table,stem,cases,period=None):
    period=period or (OBS_START,OBS_END)
    path=EXPORT/f'{stem}_{period[0]}_{period[1]}.csv'
    table.to_csv(path,index=False);register(path,stem,cases,period)
    return path

print(f'Group: {GROUP}; model diagnostics: {START}–{END}; observational skill: {OBS_START}–{OBS_END}')
print('Inputs:',BANK/GROUP,'Outputs:',EXPORT)

In [ ]:
RUN_FIA=False
RUN_ANALYTICAL=False
RUN_ROBUSTNESS=False
OBS_FIA_INTERPOLATION_VERIFIED=False
FIA_DEFINITION_VERIFIED=False
GEOMETRY_MATCH_VERIFIED=False
PROVENANCE_REVIEWED=False
FINAL_EXPORT=False
REGIONAL_PARTITION_VERIFIED=False
REGIONAL_SUM_TOLERANCE=1e-5  # 10^3 km²; meaningful only for a verified exhaustive partition.
REGULARISATION_SPEED={}  # Experiment -> verified u0 (m/s), from the executed model version.
# Choose all retained cases for final exports. Smaller exploratory selections must be labelled.
ACTIVE_EXPERIMENTS=EXPERIMENTS

## 1. Experiment and provenance record

The expected settings below come from the reviewed namelists, not directory names. `no-slip-LFI` has ellipse ratios 1.2; `no-slip-LFI-match` has 1.5. `Cs-high-def-rhe` also changes EVP subcycling to 720 and is not an isolated tensile test. The two drag-off controls retain nonzero stored Cs but disable the drag switch. No new high-tensile case is implied by the unresolved legacy label `Cs-high-ktens-high`.

I'll check the executed namelists, model/analysis commits, grid and geometry versions/checksums, forcing versions, restart lineage, spin-up, actual daily coverage and history output definitions. Multiple archived namelists require an explicit selection; an archive filename alone does not prove that it governed all history segments. The old archive dry run found no history in the three case directories, so actual job output locations still need confirming.

In [ ]:
NAMELIST_SELECTION={}  # Explicit experiment -> archived ice_in path if multiple records exist.
rows=[]
for row in manifest.to_dict('records'):
    name=row['experiment']; paths=contexts(name)['pth_cfg']; root=paths.output_root
    candidates=sorted((root/'run_metadata').glob('*/ice_in'))
    path=Path(NAMELIST_SELECTION[name]) if name in NAMELIST_SELECTION else (candidates[0] if len(candidates)==1 else None)
    row.update(output_root=str(root),history_present=(root/'history/daily').is_dir(),
        zarr_present=(root/'zarr/iceh_daily.zarr').is_dir(),namelist_path=str(path) if path else '',
        namelist_sha256='',namelist_matches=False,provenance_reviewed=False,
        model_commit='',forcing_version='',restart_lineage='',spinup='1995–1999',
        regularisation_speed=REGULARISATION_SPEED.get(name,''),configuration_note='pending or ambiguous archive metadata')
    if path is not None:
        actual=wf.namelist_values(path); mismatches=[]
        for key in ['boundary_condition','lateral_drag','cs','cq','ktens','e_yieldcurve','e_plasticpot','kstrength','ndte']:
            value=actual.get(key)
            expected=row[key]
            try:
                if isinstance(expected,bool): value=value.lower()=='.true.' if value in ['.true.','.false.'] else None
                elif isinstance(expected,(int,float)): value=float(value.lower().replace('d','e'))
                matched=value==expected
            except (AttributeError,ValueError): matched=False
            if not matched:mismatches.append(key)
        row.update(namelist_sha256=wf.file_sha256(path),namelist_matches=not mismatches,
                   configuration_note='matches expected scalar physics' if not mismatches else 'MISMATCH: '+', '.join(mismatches),
                   grid_file=actual.get('grid_file',''),form_factor_file=actual.get('f2_file',''),
                   atm_data_dir=actual.get('atm_data_dir',''),ocn_data_dir=actual.get('ocn_data_dir',''))
    rows.append(row)
manifest=pd.DataFrame(rows)
MANIFEST_PATH=EXPORT/'experiment_manifest.csv'
manifest.to_csv(MANIFEST_PATH,index=False)
provenance['manifest_sha256']=wf.file_sha256(MANIFEST_PATH)
SIGNATURE=wf.signature(provenance)
(EXPORT/'analysis_configuration.json').write_text(json.dumps(provenance,indent=2))
display(manifest)

## 2. Dates, units and definitions

I'll use exactly matched calendar-day support after checking native timestamps, leap days, missing days and the source sampling convention. Inner joins must not conceal dropped dates. The helpers reject duplicate/missing days and unreviewed units. FIA is concentration-weighted area when read from shuga; classified-cell extent is a different quantity. I'll reconcile the manuscript equation with the metric actually used.

Native AF2020 sampling is not automatically a daily observation. The optional interpolation below requires explicit review, bracketed source dates, no extrapolation and a maximum 20-day gap. It does not create independent daily observations. A comparison at native composite support requires its own interval treatment. I'll record which approach is used.

I'll check that the BS/AS bias signs and regional sums follow from the source products rather than repairing signs by eye. The previously quoted BS differences −46.3/−50.2 with observed mean 38.6 are impossible as model-minus-observation areas. The P95 difference 604.6−626.6 is −22.0 at displayed precision. Regional boundaries may have gaps/overlaps, so equality of summed sectors and the total is only expected after their partition is verified.

Centred classification needs surrounding history: at least five days for 9-of-11 and seven days for the 15-day rolling window. I'll also quantify valid exact-zero speed cells before accepting the classifier's zero-speed exclusion. The second notebook contains that audit.

## 3. Processing recipes — run through PBS, not automatically

In [ ]:
commands=[]
for name in ACTIVE_EXPERIMENTS:
    common=['--sim-name',name,'--experiment-group',GROUP,'--start-date',START,'--end-date',END,
        '--project',CFG['project'],'--user',CFG['user'],'--hemisphere','SH',
        '--afim-output-root',str(BANK)]
    classification=['python3','scripts/classification/classify.py',*common,'--ice-type','FI',
        '--grid-type','Tc','--ispd-thresh',str(CFG['ispd_thresh']),'--aice-thresh',str(CFG['aice_thresh']),
        '--methods','raw,binary-days,rolling-mean','--bin-window','11','--bin-min-days','9',
        '--roll-window','15','--skip-history-conversion']
    metrics=['python3','scripts/metrics/metrics.py',*common,'--ice-type','FI','--grid-type','Tc',
        '--ispd-thresh',str(CFG['ispd_thresh']),'--methods',METHOD,'--bin-window','11','--bin-min-days','9',
        '--metric-groups','fi_core,fi_spatial,fi_regional,fi_summary','--update-missing-only']
    commands.extend([shlex.join(classification),shlex.join(metrics)])
commands.extend([shlex.join(['python3','scripts/comparisons/FIA_statistics_tables.py',
    '--experiment-group',GROUP,'--sim-names',' '.join(ACTIVE_EXPERIMENTS),
    '--start-date',OBS_START,'--end-date',OBS_END,'--classification',METHOD,'--grid-type','Tc']),
    '# FIP spatial support/remapping is reviewed and computed in LD-performance.ipynb.'])
(EXPORT/'processing_commands.sh').write_text('\n'.join(commands)+'\n')
print(EXPORT/'processing_commands.sh')
print('Run these from the standalone shuga repository, after loading the analysis environment.')

Archive staging and NetCDF-to-Zarr conversion are separate operations. From shuga, inspect `python3 scripts/admin/archive.py --verbose`, then stage completed segments with `--send-to-archive`. The explicit case map is free-slip → free-slip-no-drag, free-slip01 → Cs-high-ktens-zero, free-slip02 → no-slip-LFI-match. Verify actual source locations first. Convert history before using `--skip-history-conversion`.

`--update-missing-only` does not validate old values after definitions, periods or source files change. I'll rebuild the affected products and record their versions rather than mixing legacy SeaIceToolbox values with new stores. I'll use actual cell areas, never the nominal 25 km fallback. Shared grid assets remain outside experiment groups.

## 4. Analytical form functions

In [ ]:
if RUN_ANALYTICAL:
    fig,ax=plt.subplots(figsize=(7,4))
    speed=np.linspace(0,1.1e-3,900)
    for name in FAMILIES['static']+FAMILIES['quadratic']:
        row=manifest.set_index('experiment').loc[name]
        if not bool(row.namelist_matches): raise ValueError(f'{name}: verify namelist first')
        coefficient=float(row.cs if name in FAMILIES['static'] else row.cq)
        if coefficient<=0: raise ValueError('Nonpositive coefficient')
        if name in FAMILIES['static']:
            u0=float(row.regularisation_speed)
            if u0<=0: raise ValueError('Verify positive static regularisation speed')
            phi=coefficient/(speed+u0)
        else:
            phi=coefficient*speed
        ax.plot(speed,phi,label=name)
    ax.axvline(5e-4,color='gray',linestyle=':',label='FI speed threshold')
    ax.set(xlabel='Ice speed (m s⁻¹)',ylabel='φ(u) (verified formulation units)',title='Static and quadratic form functions')
    ax.legend(fontsize=8); fig.tight_layout()
    path=EXPORT/'analytical_form_functions_Cs_Cq.png'
    fig.savefig(path,dpi=220)
    register(path,'analytical form functions',FAMILIES['static']+FAMILIES['quadratic'],(START,END),'Namelist coefficients; no strain-blend or linear curves')
    display(fig);plt.close(fig)

## 5. FIA: matched daily support, regional tables and climatology
Use concentration-weighted FIA consistently with `shuga`; distinguish it from classified-cell extent. This section refuses unknown units, missing dates, or negative area. It requires `FIA` and `FIA_by_region`; the observation builder provides fortnightly `FIA(time,region)`. Daily skill uses explicitly reviewed linear interpolation with no extrapolation and a 20-day maximum gap. It is interpolated observational skill, not daily independent observations. An observation-date comparison is an alternative if the AF2020 compositing intervals are explicitly accounted for. Region labels must match exactly. Check regional partitions and totals, especially BS and AS; do not repair signs by hand.

In [ ]:
def model_fia(name,method=METHOD):
    ds=load(name,'metrics',['FIA','FIA_by_region'],method)
    total=wf.area_to_kkm2(audited(ds.FIA,name))
    regional=wf.area_to_kkm2(audited(ds.FIA_by_region,name))
    if 'region' in total.dims or 'region' not in regional.dims:
        raise ValueError('Review FIA/region schema before concatenation')
    out=xr.concat([total.expand_dims(region=['circum_antarctic']),regional],dim='region')
    out.attrs=dict(total.attrs)
    return out

FIA_DATA={}; FIA_RESULTS=None
if RUN_FIA:
    if not FIA_DEFINITION_VERIFIED:
        raise ValueError('Verify FIA equation, concentration weighting and observational definition first')
    with open_product(OBS_FIA_PATH) as obs_ds:
        obs_native=obs_ds['FIA'].load()
    if not OBS_FIA_INTERPOLATION_VERIFIED:
        raise ValueError('Native FIA is fortnightly: review daily interpolation before proceeding')
    obs=wf.daily_observation(obs_native,OBS_START,OBS_END,max_gap_days=20)
    tables=[]
    for name in ACTIVE_EXPERIMENTS:
        data=model_fia(name).sel(time=slice(OBS_START,OBS_END)).compute()
        FIA_DATA[name]=data
        table=wf.fia_table(data,obs,OBS_START,OBS_END)
        table.insert(0,'experiment',name); tables.append(table)
        COVERAGE.append(dict(experiment=name,diagnostic='FIA',n_days=data.sizes['time'],status='passed'))
    FIA_RESULTS=pd.concat(tables,ignore_index=True)
    export_table(FIA_RESULTS,'FIA_skill',ACTIVE_EXPERIMENTS)
    display(FIA_RESULTS)
else:
    print('Pending: enable RUN_FIA after preflight')

In [ ]:
# Replot this cell without reopening histories or recalculating metrics.
if FIA_DATA:
    fig,ax=plt.subplots(figsize=(10,4))
    for name in [n for n in PRIMARY if n in FIA_DATA]:
        da=wf.area_to_kkm2(FIA_DATA[name].sel(region='circum_antarctic'))
        clim=da.groupby('time.month').mean('time')
        annual=da.resample(time='MS').mean('time')
        spread=annual.groupby('time.month').std('time')
        ax.plot(clim.month,clim,label=name)
        ax.fill_between(clim.month,clim-spread,clim+spread,alpha=.08)
    observed=wf.area_to_kkm2(obs.sel(region='circum_antarctic').sel(time=slice(OBS_START,OBS_END)))
    clim=observed.groupby('time.month').mean('time')
    ax.plot(clim.month,clim,'k--',label='AF2020')
    ax.set(xlabel='Month',ylabel='FIA (10³ km²)',title=f'{OBS_START}–{OBS_END}: monthly climatology')
    ax.legend(ncol=3,fontsize=8); fig.tight_layout()
    path=EXPORT/f'FIA_climatology_{OBS_START}_{OBS_END}.png'
    fig.savefig(path,dpi=220); register(path,'FIA climatology',PRIMARY,(OBS_START,OBS_END),
        'Shading: interannual monthly standard deviation; partial endpoint years')
    display(fig); plt.close(fig)

In [ ]:
# Per-year variability and seasonal extrema; partial endpoint years are explicit.
if FIA_DATA:
    annual=[]
    for name,data in FIA_DATA.items():
        da=wf.area_to_kkm2(data)
        for year in sorted(set(da.time.dt.year.values.tolist())):
            part=da.sel(time=da.time.dt.year==year)
            for region in part.region.values:
                series=part.sel(region=region)
                annual.append(dict(experiment=name,region=str(region),year=int(year),
                    n_days=series.sizes['time'],complete_calendar_year=series.sizes['time']==len(pd.date_range(f'{year}-01-01',f'{year}-12-31')),
                    mean=float(series.mean()),min=float(series.min()),max=float(series.max()),
                    peak_date=str(pd.Timestamp(series.time.values[int(series.argmax('time'))]).date()),
                    minimum_date=str(pd.Timestamp(series.time.values[int(series.argmin('time'))]).date())))
    export_table(pd.DataFrame(annual),'FIA_annual_extrema',EXPERIMENTS)

## 6. Matched effects, geometry and rheology

All differences are **target minus reference** as labelled. Boundary-only: free-slip-no-drag minus no-slip-LFI-match. Drag effect: active Cs/Cq minus free-slip-no-drag. Added tensile term: Cs-high minus Cs-high-ktens-zero, supplemented by the full 0/0.05/0.1/0.2 sequence. I'll interpret these as model contrasts and evaluate observational skill separately.

The geometry cases are matched in namelist physics apart from the form-factor source. Their responses need not add linearly under maximum-combined geometry. Zero added tensile strength retains compressive/shear rheology; the static tensile sequence does not test quadratic tensile dependence or provide a full factorial design.

In [ ]:
if FIA_RESULTS is not None:
    for label,(target,reference) in PAIRS.items():
        if target not in FIA_DATA or reference not in FIA_DATA:
            print(f'Pending {label}: {target} / {reference}');continue
        table=wf.pair_table(FIA_DATA[target],FIA_DATA[reference],OBS_START,OBS_END)
        table.insert(0,'target',target);table.insert(1,'reference',reference)
        export_table(table,'contrast_'+label,[target,reference]);display(table)
    for label,cases in {'coefficient':FAMILIES['static']+FAMILIES['quadratic'],
        'geometry':['cst-drag','GIB-drag','Cs-high'],'tensile':TENSILE_CASES,
        'other_rheology':FAMILIES['rheology']}.items():
        export_table(FIA_RESULTS[FIA_RESULTS.experiment.isin(cases)],label+'_FIA',cases)
    # Sector/total mismatch is a diagnostic, not an instruction to fix signs.
    for name,data in FIA_DATA.items():
        sectors=data.drop_sel(region='circum_antarctic');total=data.sel(region='circum_antarctic')
        table=wf.audit_region_totals(sectors,total,REGIONAL_SUM_TOLERANCE)
        export_table(table,'region_total_audit_'+name,[name])
        if REGIONAL_PARTITION_VERIFIED and not table.within_tolerance.all():
            raise ValueError(f'{name}: a verified partition does not reconcile with total area')

## 7. Seasonal growth and classification robustness

I'll compare April–July growth, retreat timing, extrema and interannual spread. Slopes through a changing FI mask are not a closed thermodynamic budget; mask-transition terms must be included before process attribution. Partial endpoint years are labelled and not silently treated as complete seasons. The supporting notebook includes occurrence/retention and daily tendency summaries.

Parameter choice and scoring on the same years give in-sample skill. I'll compare rankings across years and the raw/binary-days/rolling-mean classifications without interpreting adjacent grid pixels as independent samples.

In [ ]:
if FIA_DATA:
    slopes=[]
    for name,data in FIA_DATA.items():
        table=wf.seasonal_slopes(wf.area_to_kkm2(data.sel(region='circum_antarctic')))
        table['experiment']=name;slopes.append(table)
    export_table(pd.concat(slopes,ignore_index=True),'April_July_FIA_slope',list(FIA_DATA))
if RUN_ROBUSTNESS:
    if not RUN_FIA:raise ValueError('Run FIA first')
    tables=[]
    for method in ['raw','binary-days','rolling-mean']:
        for name in ACTIVE_EXPERIMENTS:
            data=model_fia(name,method).sel(time=slice(OBS_START,OBS_END)).compute()
            table=wf.fia_table(data,obs,OBS_START,OBS_END)
            table['experiment']=name;table['method']=method;tables.append(table)
    export_table(pd.concat(tables,ignore_index=True),'classification_robustness',ACTIVE_EXPERIMENTS)

## 8. Manuscript exports and scientific review

The companion notebook owns FIP support/remapping, mechanics, thickness and pack checks. I'll use its numerical exports alongside FIA rather than judging a closure by area alone. Regenerate the analytical form-function, FIA and mechanics placeholder figures with the retained cases. Check all eight sector maps, region names, coordinates and colour bars; the Australian-path/Bellingshausen-caption mismatch needs a consistent correction. I'll resolve the legacy Cs-high-ktens-high provenance before using its results.

I'll finish the discussion, conclusions and abstract from verified numbers, document limitations and attribution, and compile the complete TeX source with its figures, bibliography and Copernicus class. Notebook execution alone does not establish manuscript readiness. Missing thickness observations should lead to a bounded plausibility discussion, not a claim of validation. Coupled, wave/tide, Arctic, resolution and extended-period experiments remain outside this publication's current scope.

### Writing and figure checks I'll finish here

- I'll reconcile the September scope changes with the TeX, retained equations/macros, experiment table and figure list. Linear and strain-blend families, waves/tides and the other out-of-scope extensions should not remain in the scientific results by accident.
- I'll check the Australian-sector form-factor file paths against the Bellingshausen labels/statistics, and correct the actual region, geometry annotations and captions together. I'll verify that VOL and WS use their intended PNG resources.
- I'll replace the analytical form-function, FIA and mechanical comparison placeholders with checked exports. I'll preserve the intended two-panel sector comparisons, titles, coordinates and colour bars; no blanket image cropping. I'll consolidate overlapping FIP sections and move repetitive sector maps to the supplement if useful.
- I'll write the parameter trade-offs, seasonal/regional errors, geometry and forcing limitations, and implications for coupled models into the discussion. Parameter selection and evaluation on these same years remain in-sample evidence.
- I'll finish conclusions and quantitative abstract wording only after the tables reconcile. I'll check authorship, dataset attribution, funding/disclaimer, accessible code/data versions, and the final questions for co-authors. I'll assemble `copernicus.cls`, `refs_bbt.bib`, all figure resources and style dependencies, then compile and inspect the manuscript and supplement before Alex's review.

The three new contrasts are not a complete boundary × drag × tensile factorial design. The tensile sequence tests the static closure; it does not establish quadratic-closure tensile dependence. Zero added tensile strength retains other compressive/shear resistance, and low speed alone does not demonstrate mechanical coherence.

### Working abstract — wording to complete after the comparisons

Antarctic landfast sea ice is an important component of the coastal cryosphere, influencing air–sea exchange, polynya activity, coastal access and habitat. Its mechanical anchors are often unresolved in global sea-ice models. Here I introduce and evaluate a lateral-drag parameterisation in CICE6, separating geometric form factors from speed-dependent form functions and representing coastal and grounded-iceberg resistance on C-grid velocity points. Building on the coastal-drag formulation of Liu et al. (I'll check the publication year against the bibliography), I use a global quarter-degree stand-alone configuration to compare matched boundary conditions and drag activation, coefficient choices, geometric contributions and rheological sensitivity. **I'll add verified quantitative results and limitations here.** The assessment will establish how far the formulation represents Antarctic fast-ice anchoring without adding grounded icebergs to the land mask, and what remains to be tested before coupled-model use.


In [ ]:
REVIEW={
    'source_and_restart_provenance_reviewed':PROVENANCE_REVIEWED,
    'namelist_physics_matched':bool(manifest.namelist_matches.all()),
    'FIA_definition_and_recomputed_numbers':FIA_DEFINITION_VERIFIED and FIA_RESULTS is not None,
    'all_three_controls_analysed':all(n in FIA_DATA for n in ['free-slip-no-drag','no-slip-LFI-match','Cs-high-ktens-zero']),
    'classification_robustness_checked':RUN_ROBUSTNESS,
    'companion_diagnostics_reviewed':False,
    'legacy_case_identity_resolved':False,
    'figures_captions_and_regional_signs_checked':False,
    'discussion_conclusions_abstract_complete':False,
    'authorship_funding_code_data_statements_checked':False,
    'full_manuscript_compiled_and_visually_checked':False,
}
(EXPORT/'manuscript_artifacts.json').write_text(json.dumps(ARTIFACTS,indent=2))
(EXPORT/'manuscript_review.json').write_text(json.dumps({k:bool(v) for k,v in REVIEW.items()},indent=2))
display(pd.DataFrame({'item':REVIEW.keys(),'complete':REVIEW.values()}))
for ds in OPEN.values():ds.close()
OPEN.clear()
if FINAL_EXPORT and not all(REVIEW.values()):raise RuntimeError('Scientific/manuscript review is incomplete')